In [14]:
import re
import pandas as pd
from tqdm import tqdm

# ============================================================
# CONFIG
# ============================================================
INPUT_CSV = "~/code/mh-shortcut-learning/data/stage0/standardized/umd_standardized.csv"
OUTPUT_CSV = "umd_standardized_demographics.csv"

# ============================================================
# PATTERNS
# ============================================================

AGE_GENDER_PATTERNS = [
    re.compile(r'[\(\[]\s*(\d{1,3})\s*(m|f|nb|male|female|nonbinary|enby)\s*[\)\]]', re.I),
    re.compile(r'(?<![.\d])\b(\d{1,3})\s*(m|f|nb|male|female|nonbinary|enby)\b', re.I),
]

AGE_PATTERNS = [
    re.compile(r'\b(\d{1,3})\s*(yo|y/o|years?\s*old|yrs?\s*old)\b', re.I),
    re.compile(r'at my age\s*[\(\[]?\s*(\d{1,3})\s*[\)\]]?', re.I),
    re.compile(r'\b(?:i am|i\'m|im)\s+(?:actually\s+)?(\d{1,3})\b', re.I),
    re.compile(r'\b(\d{1,3})\s*-\s*(\d{1,3})\s*year old\b', re.I),
    re.compile(r'\b(\d{1,3})\s*-\s*(\d{1,3})\s*years old\b', re.I),
]

GENDER_PATTERNS = [
    re.compile(r'\b(?:i am|i\'m|im)\s+(?:a\s+)?(girl|boy|man|woman|male|female|nonbinary|nb|enby)\b', re.I),
    re.compile(r'\bbiologically\s+(male|female)\b', re.I),
    re.compile(r'\b(?:a\s+)?(detrans woman|detrans man|trans woman|trans man)\b', re.I),
]

EDUCATION_PATTERNS = [
    re.compile(r'\b(high school|secondary school)\b', re.I),
    re.compile(r"\b(college student|undergrad|undergraduate|bachelor['']?s?|b\.a\.|b\.s\.)\b", re.I),
    re.compile(r"\b(master['']?s?|m\.s\.|m\.a\.)\b", re.I),
    re.compile(r'\b(phd|ph\.d\.|doctorate|doctoral student|grad student|graduate student)\b', re.I),
    re.compile(r'\b(professor|postdoc|researcher)\b', re.I),
    re.compile(r'\b(student|freshman|sophomore|junior|senior)\b', re.I),
]

# ============================================================
# NORMALIZATION
# ============================================================

def normalize_gender(value: str):
    if not value:
        return None
    v = value.strip().lower()
    mapping = {
        "m": "male", "male": "male", "man": "male", "boy": "male",
        "f": "female", "female": "female", "woman": "female", "girl": "female",
        "nb": "nonbinary", "nonbinary": "nonbinary", "enby": "nonbinary",
        "trans man": "trans_man", "trans woman": "trans_woman",
        "detrans man": "detrans_man", "detrans woman": "detrans_woman",
    }
    return mapping.get(v, v)

def normalize_education(value: str):
    if not value:
        return None
    v = value.strip().lower()
    if v in {"high school", "secondary school"}:
        return "high_school"
    if v in {"college student", "undergrad", "undergraduate", "bachelor's", "bachelors", "b.a.", "b.s."}:
        return "undergraduate"
    if v in {"master's", "masters", "m.s.", "m.a."}:
        return "masters"
    if v in {"phd", "ph.d.", "doctorate", "doctoral student", "grad student", "graduate student"}:
        return "phd_or_grad"
    if v in {"professor", "postdoc", "researcher"}:
        return "academic_staff"
    if v in {"student", "freshman", "sophomore", "junior", "senior"}:
        return "student"
    return v

def is_valid_age(age):
    try:
        return 10 <= int(age) <= 100
    except:
        return False

# ============================================================
# SINGLE-POST EXTRACTION (unchanged logic)
# ============================================================

def extract_demographics(text):
    if pd.isna(text):
        text = ""
    text = str(text)

    # Strip quoted / reported speech
    text = re.sub(r'"[^"]*"', ' ', text)
    text = re.sub(r'\u201c[^\u201d]*\u201d', ' ', text)
    text = re.sub(r'(?m)^[ \t]*>.*$', ' ', text)

    result = {
        "age": None, "age_min": None, "age_max": None,
        "gender": None, "education_level": None,
        "matched_excerpt": None, "pattern_type": None, "confidence": "low",
    }

    # 1. Age + gender together
    for pat in AGE_GENDER_PATTERNS:
        m = pat.search(text)
        if m and is_valid_age(m.group(1)):
            result.update(age=int(m.group(1)), gender=normalize_gender(m.group(2)),
                          matched_excerpt=m.group(0), pattern_type="age_gender", confidence="high")
            return result

    # 2. Age range
    for pat in AGE_PATTERNS[-2:]:
        m = pat.search(text)
        if m and is_valid_age(m.group(1)) and is_valid_age(m.group(2)):
            result.update(age_min=int(m.group(1)), age_max=int(m.group(2)),
                          matched_excerpt=m.group(0), pattern_type="age_range", confidence="medium")
            for gpat in GENDER_PATTERNS:
                gm = gpat.search(text)
                if gm:
                    result["gender"] = normalize_gender(gm.group(1))
                    break
            for epat in EDUCATION_PATTERNS:
                em = epat.search(text)
                if em:
                    result["education_level"] = normalize_education(em.group(1))
                    break
            return result

    # 3. Age only
    for pat in AGE_PATTERNS[:-2]:
        m = pat.search(text)
        if m and is_valid_age(m.group(1)):
            result.update(age=int(m.group(1)), matched_excerpt=m.group(0),
                          pattern_type="age_only", confidence="medium")
            break

    # 4. Gender only
    for pat in GENDER_PATTERNS:
        m = pat.search(text)
        if m:
            result["gender"] = normalize_gender(m.group(1))
            if result["matched_excerpt"] is None:
                result.update(matched_excerpt=m.group(0), pattern_type="gender_only")
            result["confidence"] = "medium" if result["age"] is None else result["confidence"]
            break

    # 5. Education only
    for pat in EDUCATION_PATTERNS:
        m = pat.search(text)
        if m:
            result["education_level"] = normalize_education(m.group(1))
            if result["matched_excerpt"] is None:
                result.update(matched_excerpt=m.group(0), pattern_type="education_only")
            if result["age"] is None and result["gender"] is None:
                result["confidence"] = "medium"
            break

    return result

# ============================================================
# USER-LEVEL EXTRACTION: merge across all posts for a user
# ============================================================

def extract_user_demographics(texts):
    """Run extraction on every post for one user; merge the best result."""
    best = {"age": None, "age_min": None, "age_max": None,
            "gender": None, "education_level": None,
            "matched_excerpt": None, "pattern_type": None, "confidence": "low"}
    conf_rank = {"low": 0, "medium": 1, "high": 2}

    for text in texts:
        r = extract_demographics(text)
        # Keep highest-confidence age
        if r["age"] is not None and conf_rank.get(r["confidence"], 0) >= conf_rank.get(best["confidence"], 0):
            best["age"] = r["age"]
            best["confidence"] = r["confidence"]
            best["matched_excerpt"] = r["matched_excerpt"]
            best["pattern_type"] = r["pattern_type"]
        if r["age_min"] is not None and best["age"] is None:
            best["age_min"] = r["age_min"]
            best["age_max"] = r["age_max"]
        # Fill gender / education if missing
        if r["gender"] is not None and best["gender"] is None:
            best["gender"] = r["gender"]
        if r["education_level"] is not None and best["education_level"] is None:
            best["education_level"] = r["education_level"]

    return best

# ============================================================
# LOAD, PARSE TID, GROUP BY USER, ANNOTATE, INFLATE BACK
# ============================================================

df = pd.read_csv(INPUT_CSV)

# Parse TID
# parts = df['TID'].str.split('|')
# df['user_id'] = parts.str[0].astype(int)
df['post_id'] = df['sentence_id']
df['condition'] = df['label']
df['text'] = df['sentence']
print(f"Loaded {len(df)} posts from {df['user_id'].nunique()} users")

# Group by user_id and run extraction with tqdm
user_groups = df.groupby('user_id')['text'].apply(list)

user_demos = {}
for uid, texts in tqdm(user_groups.items(), total=len(user_groups), desc="Users", file=__import__('sys').stdout):
    user_demos[uid] = extract_user_demographics(texts)

user_demo_df = pd.DataFrame.from_dict(user_demos, orient='index')
user_demo_df.index.name = 'user_id'
user_demo_df = user_demo_df.reset_index()

# Merge back to post-level (inflate)
out = df.merge(user_demo_df, on='user_id', how='left')

# Reorder columns
out = out[['user_id', 'post_id', 'condition', 'text', 
           'age', 'age_min', 'age_max', 'gender', 'education_level',
           'matched_excerpt', 'pattern_type', 'confidence']]

out.to_csv(OUTPUT_CSV, index=False)

# Stats
print(f"\nDone. {len(out)} posts, {out['user_id'].nunique()} users")
print(f"Age found:       {(out.groupby('user_id')['age'].first().notna() | out.groupby('user_id')['age_min'].first().notna()).sum()} users")
print(f"Gender found:    {out.groupby('user_id')['gender'].first().notna().sum()} users")
print(f"Education found: {out.groupby('user_id')['education_level'].first().notna().sum()} users")
print(f"Saved to: {OUTPUT_CSV}")

out.head(10)

Loaded 21264 posts from 442 users
Users: 100%|██████████| 442/442 [00:07<00:00, 61.05it/s] 

Done. 21264 posts, 442 users
Age found:       211 users
Gender found:    79 users
Education found: 217 users
Saved to: umd_standardized_demographics.csv


,user_id,post_id,condition,text,age,age_min,age_max,gender,education_level,matched_excerpt,pattern_type,confidence
0,22941,1a2amm,at_risk,i want to get in to competitive pokémon playin...,16.0,NaN,NaN,NaN,high_school,i'm 16,age_only,medium
1,22941,1axicl,at_risk,"just wondering, is it safe for me to use Omegl...",16.0,NaN,NaN,NaN,high_school,i'm 16,age_only,medium
2,22941,1bwafu,at_risk,"I'm still afraid of the dark, help. I have bee...",16.0,NaN,NaN,NaN,high_school,i'm 16,age_only,medium
3,22941,1ddl6m,at_risk,my place in the world/universe...is it true wh...,16.0,NaN,NaN,NaN,high_school,i'm 16,age_only,medium
4,22941,1dpf5c,at_risk,I've been trying to LD for a long time For the...,16.0,NaN,NaN,NaN,high_school,i'm 16,age_only,medium
5,22941,1ds0va,at_risk,I realise that I'm actually scared to LD. I ke...,16.0,NaN,NaN,NaN,high_school,i'm 16,age_only,medium
6,22941,1etc2b,at_risk,I feel so retarded. In advance Im sorry if thi...,16.0,NaN,NaN,NaN,high_school,i'm 16,age_only,medium
7,22941,1f44i7,at_risk,Any advice on my youtubing? [Here's my channel...,16.0,NaN,NaN,NaN,high_school,i'm 16,age_only,medium
8,22941,1fmeac,at_risk,IWTL how to roll my Rs. All my brothers and ev...,16.0,NaN,NaN,NaN,high_school,i'm 16,age_only,medium
9,22941,1fn8jb,at_risk,A huge problem with FILD and sleep paralysis I...,16.0,NaN,NaN,NaN,high_school,i'm 16,age_only,medium


False Positive Samples
i was lowballing when some guy came with flawless opal gemstone there sell price were at 20m a a peice and he asked for 50m for 5. i accepted and than when i was about to sell them they drop to 2m ech then he said get sammed and left. kinda funny if being honest.	20m
i m at about 18m in i e sales and today for the first time i stole and sold a zero damage car. grabbed a sterling on the boardwalk. drove it back to la mesa. prepped it in matte black. turned off my headlights and hauled to the sale. there was an oppressor at the end of the block but they didn t seem to care about me.	18m
hello! my mom 69f is looking to buy a home in lakeside. i'm not sure if that area would be a good fit for her and her little dog. any thoughts or feelings one way or the other?	69f
you get that reward in exchange for risk, because your 10m could have become 8m or 0m. anyway, your profits are only on paper, and when you realize them, they'll become 40 smaller after taxes. also, someone with 10, rather than 10m, can buy the same etf and make the same percentage profit in an easily accessible financial system. i have many objections to our wealth-favoring tax code, but no objection to our stock markets.	10m
hallo zusammen, aktuell wohne ich mit meiner freundin in stuttgart unter absoluten traumbedingungen 70m , 850 warm, dachgeschoss, gemeinschaftsgarten, geringer weg zu fu in die stadt . allerdings ist das mietverh ltnis bis zum 01.01.2025 befristet und wir wollen im oktober raus. daher meine frage ist es realistisch f r die verbleibenden 4 monate nachmieter bzw. untermieter zu finden, um maximal einen monat doppelt zahlen zu m ssen?	70m
its usually about 50m in game or so, you ll see them swinging a sword moving back and forth.	50m
hello all, i bought my first 10g fish tank today and set it up with my local tap water which is safe with some tetra 2x chlorine remover, used sponge filter, and tetra 78 f heater. i bought a used sponge filter which i added 15 minutes after using the de-chlorination, will this start my tank cycle i also added a tiny bit of flake food ? any suggestions and tips? for stocking i m thinking 3-4 sunset platies 1-3 male to female ratio and a kuhli loach but only 1 because i don t want to much waste in such a small tank also looking for different suggestions for tank mates that i d like to be a bottom feeder to stay out of the platies way . thanks! !	78 f
perfect! laundry 11, be 35, naruto 14, fyrebird 5 total 65 8 for priority shipping insured. 73 f f or 77 g s i will box it up and send you a picture of everything and the shipping info.	73 f
dude thinks i m joking i ve got a nice worth of niche scents and occasionally pick up high rated dupes etc. i live in southern california, and the weather is never consistent. one minute it might be cold and rain, and the next be scorching hot with high winds, and some random bits of hail. summer seems to be the most consistent weather. its just hot and dry. my point is, you can rock a cologne and it will do its job, but if you re somewhere that has a mix of weather it may not perform at all. i know elixir does very well in 80 f and below. i spray just light with the elixir and then light with this one when its hot. both blend and leave a beautiful scent trail. i use cdnim parfum and spray twice with nautica voyage to take the harshness off a bit. they blend and wala. scent trail galore.	80 f
printed in 3 tiles on prusa mk3 with inset roads, trails, streams and river. generated from 10m elevation model data and osm for inset features.	10m
we're talking about possible reasons why rey wouldn't be here next year. i am not saying i think some team will swoop in for 20m, but i do think heath would reject anyone who offered 10m. so if he's not here next year, it's because we were made an offer we couldn't refuse.	20m
i got dinged for rwt on my main. other than the simple don't rwt , does anyone have any tips or tricks to not get banned or caught again? it was 50m glad.	50m
i have several hobbies. only problem is i can t do any of them when it s 20f outside and snowing!	20f
i've 35m recently had to move in to a tiny room on my own, and it's my first time being on my own in 16 years so it's a tough one for me. i've got a small double bed, tv unit, chest of drawers and a desk crammed in here at the minute but i feel like there's no separation from the sleep and work areas so i'm finding it really hard to get any work done. i'd debated putting an ikea kallax as a divider between the desk and bed but i'm not sure. any suggestions would be appreciated! reply share.	35m

In [1]:
import pandas as pd
df = pd.read_csv('mindset-strong-gender-annot.csv')
# sample 100 random posts
sample = df.sample(100, random_state=42)
sample.to_csv('mindset-strong-gender-annot-sample.csv', index=False)

In [ ]:
out = pd.read_csv('cssrs_standardized_demographics.csv')
out.head()

,TID,text,created_utc,age,age_min,age_max,gender,education_level,matched_excerpt,pattern_type,confidence
0,899864|adhd|xaz4ls,i am currently on 98 completion of this game. ...,1.662842e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,low
1,899864|adhd|rzbzih,hello! i want to try and seek some advice rega...,1.641681e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,low
2,899864|adhd|r7eual,update she is doing so much better today. thin...,1.638472e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,low
3,899864|adhd|qczuys,so i am playing the career mode and i am curre...,1.634846e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,low
4,899864|adhd|lfca3p,hi everyone. i'm at a bit of a loss really. my...,1.612793e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,low


In [ ]:

df[df['gender'].notna()]['matched_excerpt'].value_counts().head(20)


matched_excerpt
100m             4539
10m              3813
20m              3357
a trans woman    3267
30m              3126
50m              2515
a trans man      2436
i'm a woman      2330
15m              1989
25m              1988
trans woman      1484
40m              1468
20f              1457
i am a woman     1397
23f              1360
22f              1324
25f              1298
21f              1200
24f              1185
24m              1185
Name: count, dtype: int64

In [20]:
# Keep only high-confidence age+gender disclosures (e.g. "24m", "23f", "(26f)")
# and filter to male/female only
strong = df[(df['confidence'] == 'high') & (df['gender'].isin(['male', 'female']))].copy()

# parse user_id from TID 
strong['user_id'] = strong['TID'].str.split('|').str[0].astype(int)
# ensure unique user ids
strong = strong.drop_duplicates(subset=['user_id'])
# Get unique users per gender
males   = strong[strong['gender'] == 'male']['user_id'].unique()
females = strong[strong['gender'] == 'female']['user_id'].unique()
print(f"Males: {len(males)} users, Females: {len(females)} users")

# Balance: downsample the larger group to match the smaller
n = min(len(males), len(females))
males_sample   = pd.Series(males).sample(n, random_state=42).values
females_sample = pd.Series(females).sample(n, random_state=42).values

balanced = strong[strong['user_id'].isin(set(males_sample) | set(females_sample))]
print(f"Balanced: {n} users per gender, {len(balanced)} total posts")
print(balanced['gender'].value_counts())
balanced.head()

Males: 26293 users, Females: 16100 users
Balanced: 16100 users per gender, 32200 total posts
gender
female    16100
male      16100
Name: count, dtype: int64


,TID,text,created_utc,age,age_min,age_max,gender,education_level,matched_excerpt,pattern_type,confidence,user_id
99,822003|adhd|s22moj,stupid 1st world stuff here but putting it for...,1.641981e+09,32.0,NaN,NaN,female,NaN,32f,age_gender,high,822003
441,791323|adhd|nbijqe,i am 28f but was born with or developed severa...,1.620918e+09,28.0,NaN,NaN,female,NaN,28f,age_gender,high,791323
765,318905|adhd|glrhhf,long post. . . tl dr is my sis wasn't there fo...,1.589764e+09,34.0,NaN,NaN,female,NaN,34f,age_gender,high,318905
1813,516112|adhd|kezfv4e,34 f here. never wanted kids. there was a peri...,1.703598e+09,34.0,NaN,NaN,female,NaN,34 f,age_gender,high,516112
2999,692720|adhd|h7dzak3,thank you sooo much this looks great! ! and on...,1.627874e+09,32.0,NaN,NaN,female,NaN,32f,age_gender,high,692720


In [25]:
balanced.drop(columns=['age_max', 'age_min', 'education_level', 'pattern_type', 'confidence']).to_csv('mindset-strong-gender-annot.csv', index=False)